In [ ]:
# Cell 1 - Clone repository
%cd /content
!if [ ! -d VDT_GD2 ]; then git clone https://github.com/Darkin72/VDT_GD2.git; fi
%cd /content/VDT_GD2
!git pull


In [ ]:
# Cell 2 - Paper training configuration
DEVICE = "cuda"
WEIGHT_DECAY = 1e-4
MIN_LR = 1e-6
PATIENCE = 50
MULTI_SCALE = "64, 128, 256"
AUGMENTATION = "flip H/V, rotation 0/90/180/270, brightness/contrast +/-0.1"

CONFIG = {
    # Paper: synthetic datasets use 1000 epochs, batch 16 and cosine annealing.
    "SOTS_ITS": {"epochs": 1000, "batch": 16, "lr": 2e-4, "scheduler": "cosine"},
    "SOTS_OTS": {"epochs": 1000, "batch": 16, "lr": 1e-4, "scheduler": "cosine"},
    # I-HAZE/O-HAZY are real-world datasets; use the paper's real-world settings.
    "I_HAZE": {"epochs": 500, "batch": 8, "lr": 2e-4, "scheduler": "step"},
    "O_HAZY": {"epochs": 500, "batch": 8, "lr": 2e-4, "scheduler": "step"},
}
for dataset, cfg in CONFIG.items():
    print(dataset, cfg)
print("device:", DEVICE, "| weight_decay:", WEIGHT_DECAY, "| min_lr:", MIN_LR)
print("multi-scale:", MULTI_SCALE, "| augmentation:", AUGMENTATION)


In [ ]:
# Cell 3 - Train with the paper settings
!pip -q install pillow matplotlib numpy
%cd /content/VDT_GD2

c = CONFIG["I_HAZE"]
!python -m solution.wavelet_dehaze.train "dataset/I-HAZE/train" --val-data "dataset/I-HAZE/val" --device {DEVICE} --epochs {c["epochs"]} --batch-size {c["batch"]} --size 256 --lr {c["lr"]} --weight-decay {WEIGHT_DECAY} --scheduler {c["scheduler"]} --step-size 100 --min-lr {MIN_LR} --augment --multi-scale --patience {PATIENCE} --out haze_wavelet_i_haze.pt --history history_i_haze.json

c = CONFIG["O_HAZY"]
!python -m solution.wavelet_dehaze.train "dataset/O-HAZY/train" --val-data "dataset/O-HAZY/val" --device {DEVICE} --epochs {c["epochs"]} --batch-size {c["batch"]} --size 256 --lr {c["lr"]} --weight-decay {WEIGHT_DECAY} --scheduler {c["scheduler"]} --step-size 100 --min-lr {MIN_LR} --augment --multi-scale --patience {PATIENCE} --out haze_wavelet_o_hazy.pt --history history_o_hazy.json

SOTS_ROOT = "dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]"

c = CONFIG["SOTS_ITS"]
!python -m solution.wavelet_dehaze.train "{SOTS_ROOT}/indoor/train" --val-data "{SOTS_ROOT}/indoor/val" --device {DEVICE} --epochs {c["epochs"]} --batch-size {c["batch"]} --size 256 --lr {c["lr"]} --weight-decay {WEIGHT_DECAY} --scheduler {c["scheduler"]} --min-lr {MIN_LR} --augment --multi-scale --patience {PATIENCE} --out haze_wavelet_sots_its.pt --history history_sots_its.json

c = CONFIG["SOTS_OTS"]
!python -m solution.wavelet_dehaze.train "{SOTS_ROOT}/outdoor/train" --val-data "{SOTS_ROOT}/outdoor/val" --device {DEVICE} --epochs {c["epochs"]} --batch-size {c["batch"]} --size 256 --lr {c["lr"]} --weight-decay {WEIGHT_DECAY} --scheduler {c["scheduler"]} --min-lr {MIN_LR} --augment --multi-scale --patience {PATIENCE} --out haze_wavelet_sots_ots.pt --history history_sots_ots.json


In [ ]:
# Cell 4 - Preview loss, PSNR and SSIM
import json
from pathlib import Path
import matplotlib.pyplot as plt

files = {
    "I-HAZE": "history_i_haze.json", "O-HAZY": "history_o_hazy.json",
    "SOTS-ITS": "history_sots_its.json", "SOTS-OTS": "history_sots_ots.json",
}
histories = {name: json.loads(Path(path).read_text()) for name, path in files.items() if Path(path).exists()}
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for name, rows in histories.items():
    epoch = [r["epoch"] for r in rows]
    for ax, train_key, val_key in zip(
        axes,
        ("train_loss", "train_psnr", "train_ssim"),
        ("val_loss", "val_psnr", "val_ssim"),
    ):
        ax.plot(epoch, [r[train_key] for r in rows], label=f"{name} train")
        ax.plot(epoch, [r[val_key] for r in rows], "--", label=f"{name} val")
for ax, title in zip(axes, ("Loss", "PSNR (dB)", "SSIM")):
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.grid(alpha=.25); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("haze_wavelet_training_metrics.png", dpi=160); plt.show()
